In [1]:
import calendar
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import locale
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules,apriori
from scipy import sparse

In [25]:
df = pd.read_csv('00. Review.csv')

# df['Index'] = [i for i,v in enumerate(df.Merk)]

# df = df[["Index",'Merk',"Reviewer"]]
# df = df[['Product']]
# df = df.set_index('index_name')

df

,Product,UserName,SkinCond_Age,Recommend,PostDate,Review,Rating
0,Perfect 3D Gel,MALA_,35 - 39,MALA_ recommends this product!,12 Jul 2020,Di aku beneran kerja. Melembabkan bgt. Tp gak ...,5
1,Perfect 3D Gel,Vitamaylinda,"Normal, 19 - 24",Vitamaylinda doesn't recommend this product!,31 May 2020,ga cocok bgttt bikin muka bruntusan dan lgsg m...,2
2,Perfect 3D Gel,aishdwie,"Combination, 19 - 24",aishdwie doesn't recommend this product!,30 Jul 2020,[SOLD]baru pertama kali nyoba dan beli karna l...,2
3,Perfect 3D Gel,ilmisaptiah,"Dry, 30 - 34",ilmisaptiah doesn't recommend this product!,12 Jul 2020,"enak sih di mukaku, yg tadinya kering jd lemba...",3
4,Perfect 3D Gel,alvinadin,"Oily, 19 - 24",alvinadin recommends this product!,15 Jul 2020,"SOLD OUT [PRELOVED] Halo aku preloved ini ya, ...",3
...,...,...,...,...,...,...,...
8641,Face Tonic,nurrav,"Combination, 19 - 24",nurrav recommends this product!,24 Aug 2020,aku suka bgt sama toner ini.. aku juga selalu ...,5
8642,Face Tonic,ireneflorencia,"Normal, 19 - 24",ireneflorencia recommends this product!,20 Aug 2020,Aku sempet pake ini pas jaman sd-smp dulu kare...,4
8643,Face Tonic,destinawahyuni,"Combination, 19 - 24",destinawahyuni doesn't recommend this product!,18 Aug 2020,"Honestly ini udah lama aku punya nya, tapi bar...",2
8644,Face Tonic,brigittac,"Dry, 18 and Under",brigittac recommends this product!,13 Aug 2020,toner ter murah dan aman pula. cocok bgt buat ...,5


In [24]:
df['UserName'].value_counts()

moonchild_dby       47
Yunitaella          32
lismagiovani        22
rahmadhess          21
nysrinaw            20
                    ..
Nathalianathal18     1
akutara              1
Upiaaak              1
oktinovianti         1
Farahnajela          1
Name: UserName, Length: 5281, dtype: int64

In [4]:
df['Product'].value_counts()

Face Mist                          70
Milk Cleanser                      60
Brightening Night Cream            60
Real Nature Mask Sheet             50
Pore Pack                          50
                                   ..
Optibrow Enhancing Serum            2
Lipocils Platinum Eyelash Serum     2
A-Lash Wink Serum                   1
DHC 3 in 1 eyelash serum            1
Miss Thique Brow and Lash Serum     1
Name: Product, Length: 802, dtype: int64

In [5]:
df.isna().sum()


Product           0
UserName          0
SkinCond_Age     77
Recommend       972
PostDate          0
Review            0
Rating            0
dtype: int64

In [6]:
# df = df.dropna()

In [7]:
df.Product.value_counts()

Face Mist                          70
Milk Cleanser                      60
Brightening Night Cream            60
Real Nature Mask Sheet             50
Pore Pack                          50
                                   ..
Optibrow Enhancing Serum            2
Lipocils Platinum Eyelash Serum     2
A-Lash Wink Serum                   1
DHC 3 in 1 eyelash serum            1
Miss Thique Brow and Lash Serum     1
Name: Product, Length: 802, dtype: int64

In [8]:
item_count_pivot = df.pivot_table(index='UserName',columns='Product', values= 'Rating',aggfunc='sum').fillna(0)
item_count_pivot[item_count_pivot > 0] =1
item_count_pivot.head(10)

Product,100% Organic Cold-Pressed Rose Hip Seed Oil,2.5,3 Step Nose Clear Pad,30 Seconds Meltaway Balm,7 Days Mask,99% Jeju Fresh Aloe Soothing Gel,A-Lash Wink Serum,ACNE CLEANSER SCRUB BETA PLUS,ACNE FACIAL PEELING,AHA 30% + BHA 2% Peeling Solution,...,aloe vera soothing gel 95%,ceraVe Moisturizing Cream,hutmun gel,nourish Acne plast,pulchra.id,qiansoto peel off masque,real nature mask sheet,viva milk cleanser dan viva face tonic,viva skinfood,wardah aloe vera multifunction gel
UserName,,,,,,,,,,,,,,,,,,,,,
15popipu,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27red,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28zura,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39FeraTan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABstyle_,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
APRILYANIA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AYUKSM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Aandreas,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Abhasper,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Keterangan Olah Dataset
print('Dimensi Dataset :',item_count_pivot.shape)
print('Jumlah Reviewer Item :',item_count_pivot.shape[0])
print('Jumlah Item :',item_count_pivot.shape[1])

Dimensi Dataset : (5281, 802)
Jumlah Reviewer Item : 5281
Jumlah Item : 802


In [13]:
freq = apriori(item_count_pivot, min_support= 0.001, use_colnames= True)
freq.sort_values('support',ascending=False)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:111: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
403,0.010225,(Milk Cleanser)
224,0.007385,(Face Tonic)
505,0.007196,(Pore Pack)
388,0.006817,(Masquerade Face Mask)
551,0.006059,(Real Nature Mask Sheet)
...,...,...
548,0.001326,(Raspberry Roots Depuffing Eye Gel)
44,0.001326,(Advanced Night Repair Eye Concentrate)
262,0.001326,(Game Changer Tripeptide Eye Concentrate Gel)
212,0.001136,(Eyelash Nourishing Essence)


In [30]:
rules = association_rules(freq,metric='lift',min_threshold=1)[['antecedents','consequents','support','confidence','lift']]
rules.sort_values(by=['support','confidence','lift'],ascending=False).head(20)

,antecedents,consequents,support,confidence,lift
0,(Face Tonic),(Milk Cleanser),0.002083,0.282051,27.583571
1,(Milk Cleanser),(Face Tonic),0.002083,0.203704,27.583571
4,(Nature Daily Witch Hazel Purifying Moisturize...,(Nature Daily Aloe Hydramild Facial Wash),0.001894,1.000000,528.100000
5,(Nature Daily Aloe Hydramild Facial Wash),(Nature Daily Witch Hazel Purifying Moisturize...,0.001894,1.000000,528.100000
2,(Home Peeling 2),(Home Peeling 1),0.001704,0.900000,475.290000
3,(Home Peeling 1),(Home Peeling 2),0.001704,0.900000,475.290000
